# Dask on amlhpc

Run a distributed Dask cluster on Azure Machine Learning with the built-in amlhpc dask commands: a scheduler on this Compute Instance (`dask-scheduler-up`), a pool of AmlCompute worker VMs (`dask-up`), and a clean teardown (`dask-down`). See the [README](README.md) for the full walkthrough.

In **every terminal** you use, first connect it to this workspace by sourcing the cluster profile written by `deploy init`:

```bash
source ~/.amlhpc/<clustername>.sh   # restores SUBSCRIPTION, CI_RESOURCE_GROUP, CI_WORKSPACE
sinfo                               # confirm the partitions, e.g. f4s
```

## 1. Start the scheduler (terminal 1, on the CI)

```bash
dask-scheduler-up
```

It stays in the foreground and prints the address workers and client both use, e.g. `tcp://10.0.1.5:8786`. Leave it running and copy that address.

## 2. Add worker VMs (terminal 2, on the CI)

```bash
dask-up -p f4s -s tcp://10.0.1.5:8786 --session pi-demo -N 4
```

This submits 4 worker VMs on the `f4s` partition, tagged with the session id `pi-demo`, and returns immediately. Watch them register in the client widget below before running the calculation.

## 3. Connect a client to the running scheduler

Use the `tcp://` address that `dask-scheduler-up` printed.

In [ ]:
from dask.distributed import Client, progress
client = Client("tcp://10.0.1.5:8786")   # <-- the address dask-scheduler-up printed
client

## A real calculation: estimate π with distributed Monte Carlo

Throw random darts at the unit square; the fraction landing inside the quarter circle approximates π/4. This is embarrassingly parallel, so every worker VM does an independent chunk and we reduce the counts. The more samples (and workers) you add, the closer the result gets to π.

In [ ]:
import random

def count_inside(n_samples, seed):
    rng = random.Random(seed)
    inside = 0
    for _ in range(n_samples):
        x, y = rng.random(), rng.random()
        if x * x + y * y <= 1.0:
            inside += 1
    return inside

In [ ]:
%%time

n_tasks = 512
samples_per_task = 2_000_000
total_samples = n_tasks * samples_per_task

futures = [client.submit(count_inside, samples_per_task, seed) for seed in range(n_tasks)]
progress(futures)

inside_total = sum(client.gather(futures))
pi_estimate = 4.0 * inside_total / total_samples

print(f"samples : {total_samples:,}")
print(f"pi ~    : {pi_estimate}")
print(f"error   : {abs(pi_estimate - 3.141592653589793):.2e}")

## 4. Tear down

Disconnect the client here, then stop the worker session from terminal 2 and the scheduler in terminal 1. `dask-down` cancels the worker pool so the AmlCompute VMs scale to zero and you stop paying for them.

```bash
dask-down --session pi-demo    # terminal 2
# then press Ctrl-C in terminal 1 to stop the scheduler
```

In [ ]:
client.close()